# Lab 9: tracing and cost

Build a small tracer: every model call becomes a span with its tokens, cost and latency,
spans nest into a tree, personal data is masked before anything is written, and a budget
guard refuses a call before it could pass a cap. This notebook follows
[Lab 9 in the book](https://build-ai-handbook.vercel.app/lab-tracing.html): read each step there, then run it here. The
code is in `labs/09-tracing/tracing.py`. On the fake backend every latency is tiny: read
the shape of the tree and the token counts.

## Set up

Run the next cell first. It finds the book's code (in Colab it downloads it), installs what
the labs need and picks who answers:

- `fake` needs no key and costs nothing. Its replies are scripted, so they show how the
  code works, not what a model would say.
- `anthropic` uses Claude with your own API key. In Colab, add `ANTHROPIC_API_KEY` in the
  Secrets panel (the key icon on the left) and let this notebook read it. Elsewhere the cell
  asks for the key. It is never written into the notebook.
- `ollama` runs a small open model, free. In Colab it installs and starts Ollama for you:
  switch to a GPU first (Runtime, then Change runtime type, then T4 GPU), and expect the
  first run to download about 2 GB.

To change the backend later, change it here and run this cell again. Lab 0 in the book
explains the choices: https://build-ai-handbook.vercel.app/lab-setup.html

In [ ]:
# @title Set up this lab
BACKEND = "fake"  # @param ["fake", "anthropic", "ollama"]
MODEL = ""  # @param {type:"string"}

import os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

REPO = "https://github.com/aggarwaldev/build-ai-handbook.git"


def find_root():
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "labs" / "common" / "llm.py").is_file():
            return folder
    return None


root = find_root()
if root is None:
    if not Path("build-ai-handbook").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO], check=True)
    root = Path("build-ai-handbook").resolve()
os.chdir(root)


def run_command(*cmd):
    """Run a command from the repository root, show its output in this cell, and stop if it fails."""
    done = subprocess.run([sys.executable if part == "python" else part for part in cmd],
                          capture_output=True, text=True)
    print(done.stdout, end="")
    print(done.stderr, end="", file=sys.stderr)
    if done.returncode:
        raise RuntimeError("this command failed: " + " ".join(cmd))


if str(root / "labs") not in sys.path:
    sys.path.insert(0, str(root / "labs"))

try:
    import anthropic, numpy  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "labs/requirements.txt"], check=True)

os.environ["LLM_BACKEND"] = BACKEND
if MODEL.strip():
    os.environ["LLM_MODEL"] = MODEL.strip()
else:
    os.environ.pop("LLM_MODEL", None)
IN_COLAB = "google.colab" in sys.modules

if BACKEND == "anthropic" and not os.environ.get("ANTHROPIC_API_KEY", "").strip():
    if IN_COLAB:
        from google.colab import userdata
        try:
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        except Exception as err:
            raise RuntimeError("Add ANTHROPIC_API_KEY in the Secrets panel (the key icon on the left) "
                               "and allow this notebook to read it, then run this cell again.") from err
    else:
        from getpass import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key (not saved in the notebook): ")


def ollama_running():
    host = os.environ.get("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
    try:
        urllib.request.urlopen(host + "/api/tags", timeout=2)
        return True
    except OSError:
        return False


if BACKEND == "ollama":
    if not ollama_running():
        if not IN_COLAB:
            raise RuntimeError("Start Ollama on this machine (ollama serve), or pick another backend.")
        if shutil.which("ollama") is None:
            if shutil.which("zstd") is None:
                subprocess.run("apt-get -qq update && apt-get -qq install -y zstd", shell=True, check=True)
            subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "a"), stderr=subprocess.STDOUT)
        for _ in range(60):
            if ollama_running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("Ollama did not start; see ollama.log.")
    if shutil.which("nvidia-smi") is None:
        print("No GPU found: Ollama will answer slowly on the CPU.")
    pulled = os.environ.get("LLM_MODEL", "llama3.2")
    if shutil.which("ollama"):
        subprocess.run(["ollama", "pull", pulled], check=True)

print(f"Ready: code in {root}, backend {BACKEND}" + (f", model {os.environ['LLM_MODEL']}" if os.environ.get("LLM_MODEL") else ""))

## Step 1: a span is a context manager

Open two nested spans with `with`, and read the file the tracer wrote. Each line is one
span; the inner one names the outer one as its parent. This writes to a temporary folder.

In [ ]:
import json, tempfile
sys.path.insert(0, str(root / "labs" / "09-tracing"))
import tracing
from common import llm

folder = Path(tempfile.mkdtemp())
with tracing.Tracer("notebook", out_dir=folder) as t:
    with t.span("outer"):
        with t.span("inner", note="hello"):
            pass
for line in t.path.read_text().splitlines():
    rec = json.loads(line)
    print(rec["name"], "| span", rec["span_id"], "| parent", rec["parent_id"], "|", rec["status"])

## Step 2: every model call becomes a span

Make a model call inside a span, without telling the tracer. The listener records it, and the
span carries the fields the page lists.

In [ ]:
with tracing.Tracer("one-call", out_dir=folder) as t:
    with t.span("ask"):
        llm.chat([{"role": "user", "content": "Say hello"}], max_tokens=64, effort="low")
spans = tracing.load_trace(t.path)
call = [s for s in spans if s["kind"] == "llm"][0]
print(json.dumps(call["attrs"], indent=2))

## Step 3: one JSON line per span

The file is plain JSON lines, so you can search it with `grep`. Children are written before
their parents, because a span is written when it ends.

In [ ]:
print([s["name"] for s in spans])
print(t.path.read_text().splitlines()[0][:200], "...")

## Step 4: mask before you write

`scrub` replaces values stored under sensitive key names whole, and runs a detector over
every other string. Note that `input_tokens` survives: keys are matched exactly.

In [ ]:
print(tracing.scrub({
    "email": "sam@example.com",
    "input_tokens": 48,
    "note": "Call me on 07700 900123 or write to sam@example.com",
}))

## Step 5: summarise a run

`percentile` uses the nearest-rank method. The page's worked example: five latencies give a
p50 of 0.3 and a p95 of 0.5.

In [ ]:
latencies = [0.5, 0.1, 0.4, 0.2, 0.3]
print(tracing.percentile(latencies, 50), tracing.percentile(latencies, 95))

## Step 6: a budget guard that fails closed

Ask the guard what one call could cost at worst, then try a cap that is too small. The guard
raises before the call is made.

In [ ]:
guard = tracing.BudgetGuard(cap_usd=0.0001)
messages = [{"role": "user", "content": "Plan a trip " * 20}]
print("worst case: $%.6f" % guard.worst_case(messages, None, 1024, "claude-haiku-5-5"))
try:
    guard.check(messages, None, 1024, "claude-haiku-5-5")
except tracing.BudgetExceeded as err:
    print("refused:", err)

## Step 7: trace a run

The page's demo: a small travel planner in four model calls. Read the tree top down; each
model call hangs under the step that made it.

In [ ]:
!LLM_BACKEND=fake python labs/09-tracing/tracing.py demo

Now the same demo with a cap of $0.001. The guard stops the run before the fourth call.

In [ ]:
!LLM_BACKEND=fake python labs/09-tracing/tracing.py demo --cap 0.001

Summarise the most recent trace file:

In [ ]:
!python labs/09-tracing/tracing.py summary

## Run the tests

In [ ]:
!cd labs && python -m pytest -q -p no:cacheprovider 09-tracing

## Experiments

Try the experiments in the [Experiments section](https://build-ai-handbook.vercel.app/lab-tracing.html#experiments). A quick
one: find the smallest cap that lets the whole demo finish, by editing the numbers below.

In [ ]:
for cap in (0.0005, 0.001, 0.002, 0.005):
    print("cap", cap)
    out = subprocess.run([sys.executable, "labs/09-tracing/tracing.py", "demo", "--cap", str(cap)],
                         check=True, stdout=subprocess.PIPE, text=True).stdout
    print("  ", [l for l in out.splitlines() if l.startswith(("calls:", "stopped:"))])

Then tick the lab's checklist in the book: your progress is kept in your browser, not in
this notebook.